In [2]:
from pathlib import Path
import pandas as pd

# 1. Tìm thư mục dự án
current = Path.cwd().resolve()

PROJECT_DIR = next(
    (p for p in [current, *current.parents]
     if (p / "results/llm/reproduction/generation_run_01.csv").exists()),
    None
)

assert PROJECT_DIR is not None, "Không tìm thấy dự án"

# 2. Dữ liệu đầu vào
DATA_DIR = PROJECT_DIR / "data/ACME_Insurance"

DDL_FILE = DATA_DIR / "DDL/ACME_small.ddl"
OWL_FILE = DATA_DIR / "ontology/insurance.ttl"

BASELINE_DIR = PROJECT_DIR / "results/llm/reproduction"
OUT_DIR = PROJECT_DIR / "results/extended_analysis/context_ablation"

OUT_DIR.mkdir(parents=True, exist_ok=True)

# 3. Cấu hình thí nghiệm
MODEL = "gpt-5.4-2026-03-05"
RUN_IDS = range(1, 6)
METHODS = ["SQL", "SPARQL"]

# 4. Kiểm tra dữ liệu
assert DDL_FILE.exists()
assert OWL_FILE.exists()

for run_id in RUN_IDS:
    file = BASELINE_DIR / f"generation_run_{run_id:02d}.csv"
    assert file.exists(), f"Missing baseline: {file.name}"

baseline = pd.read_csv(
    BASELINE_DIR / "generation_run_01.csv",
    keep_default_na=False
)

assert len(baseline) == 86

print("=== CONTEXT ABLATION ===")
print("Model:", MODEL)
print("Questions:", baseline["ID"].nunique())
print("Methods:", METHODS)
print("Baseline runs:", len(RUN_IDS))
print("Output:", OUT_DIR)
print("Environment: OK")

=== CONTEXT ABLATION ===
Model: gpt-5.4-2026-03-05
Questions: 43
Methods: ['SQL', 'SPARQL']
Baseline runs: 5
Output: C:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\extended_analysis\context_ablation
Environment: OK


In [3]:
import re
from rdflib import Graph, RDF, RDFS, OWL

# 1. Đọc DDL và OWL
ddl_text = DDL_FILE.read_text(encoding="utf-8-sig")

ontology = Graph()
ontology.parse(OWL_FILE, format="turtle")

# 2. Tách các CREATE TABLE theo cấu trúc DDL thực tế
pattern = r"(?im)^[ \t]*CREATE[ \t]+TABLE[ \t]+([^\s(]+)"

matches = list(re.finditer(pattern, ddl_text))
tables = {}

for i, match in enumerate(matches):
    name = match.group(1).split(".")[-1].strip('"`[]')

    start = match.start()
    end = (
        matches[i + 1].start()
        if i + 1 < len(matches)
        else len(ddl_text)
    )

    tables[name] = ddl_text[start:end].strip()

# Kiểm tra
assert len(tables) == 13, (
    f"Expected 13 tables, got {len(tables)}"
)

print("=== SQL DDL PARSING ===")
print("Số bảng trong DDL:", len(tables))
print("Danh sách bảng:", list(tables))
print("\nParsing: OK")

# 3. Tách từ để so khớp tên
def words(text):
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", str(text))
    return set(re.findall(r"[a-z]+", text.lower()))

def local_name(uri):
    return str(uri).split("#")[-1].split("/")[-1]

# 4. Chọn SQL schema
def select_sql_context(question):
    q = words(question)

    scores = {
        name: 3 * len(q & words(name)) + len(q & words(ddl))
        for name, ddl in tables.items()
    }

    selected = sorted(scores, key=scores.get, reverse=True)
    selected = [name for name in selected if scores[name] > 0][:4]

    if not selected:
        return ddl_text, len(tables), True

    # Bổ sung bảng được tham chiếu qua FOREIGN KEY
    for name in list(selected):
        refs = re.findall(
            r"REFERENCES\s+([A-Za-z_][A-Za-z_0-9]*)",
            tables[name], re.I
        )

        for ref in refs:
            if ref in tables and ref not in selected:
                selected.append(ref)

    context = "\n\n".join(tables[name] for name in selected)
    return context, len(selected), False

# 5. Chọn OWL Ontology
classes = set(ontology.subjects(RDF.type, OWL.Class))
obj_props = set(ontology.subjects(RDF.type, OWL.ObjectProperty))
data_props = set(ontology.subjects(RDF.type, OWL.DatatypeProperty))

def select_owl_context(question):
    q = words(question)

    selected = {
        cls for cls in classes
        if q & words(local_name(cls))
    }

    if not selected:
        return OWL_FILE.read_text(encoding="utf-8"), len(classes), True

    # Giữ các quan hệ kết nối Class
    for prop in obj_props:
        domain = ontology.value(prop, RDFS.domain)
        range_ = ontology.value(prop, RDFS.range)

        if domain in selected or range_ in selected:
            selected.update(x for x in (prop, domain, range_) if x is not None)

    # Giữ thuộc tính dữ liệu của Class đã chọn
    for prop in data_props:
        if ontology.value(prop, RDFS.domain) in selected:
            selected.add(prop)

    subgraph = Graph()

    for s, p, o in ontology:
        if s in selected:
            subgraph.add((s, p, o))

    n_classes = len(classes & selected)
    return subgraph.serialize(format="turtle"), n_classes, False

# 6. Chuẩn bị context cho 43 câu hỏi
contexts = {}
summary = []

questions = baseline[baseline["Method"] == "SQL"]

for _, row in questions.iterrows():
    qid = int(row["ID"])
    question = row["Question"]

    sql_ctx, n_tables, sql_fallback = select_sql_context(question)
    owl_ctx, n_classes, owl_fallback = select_owl_context(question)

    contexts[qid] = {
        "SQL": sql_ctx,
        "SPARQL": owl_ctx
    }

    summary.append({
        "ID": qid,
        "SQL Tables": n_tables,
        "OWL Classes": n_classes,
        "SQL Fallback": sql_fallback,
        "OWL Fallback": owl_fallback
    })

# 7. Kiểm tra
context_summary = pd.DataFrame(summary)

assert len(contexts) == 43

print("=== CONTEXT SELECTION ===")
print("Questions:", len(contexts))
print("SQL full-context fallbacks:", context_summary["SQL Fallback"].sum())
print("OWL full-context fallbacks:", context_summary["OWL Fallback"].sum())

display(context_summary.head(10))

context_summary.to_csv(
    OUT_DIR / "context_selection_summary.csv", index=False
)


=== SQL DDL PARSING ===
Số bảng trong DDL: 13
Danh sách bảng: ['Claim', 'Claim_Amount', 'Loss_Payment', 'Loss_Reserve', 'Expense_Payment', 'Expense_Reserve', 'Claim_Coverage', 'Policy_Coverage_Detail', 'Policy', 'Policy_Amount', 'Agreement_Party_Role', 'Premium', 'Catastrophe']

Parsing: OK
=== CONTEXT SELECTION ===
Questions: 43
SQL full-context fallbacks: 2
OWL full-context fallbacks: 2


,ID,SQL Tables,OWL Classes,SQL Fallback,OWL Fallback
0,1,4,11,False,False
1,2,5,7,False,False
2,3,4,11,False,False
3,4,4,11,False,False
4,5,7,11,False,False
5,6,6,7,False,False
6,7,4,11,False,False
7,8,5,7,False,False
8,9,4,11,False,False
9,10,7,11,False,False


In [4]:
# 4. Giữ nguyên cấu trúc prompt zero-shot của baseline

import json

# Đọc notebook baseline để tái sử dụng các hàm Kaggle
nb_file = PROJECT_DIR / "notebook/04_local_llm.ipynb"

with nb_file.open("r", encoding="utf-8") as f:
    nb = json.load(f)

print("Baseline notebook loaded:", len(nb["cells"]), "cells")


def build_sql_prompt_ablation(question, context):
    return (
        "Given the database described by the following DDL:\n"
        f"{context}\n"
        "Write a SQL query that answers the following question. "
        "Do not explain the query. return just the query, "
        "so it can be run verbatim from your response.\n"
        "Here’s the question:\n"
        f'"{question}"'
    )


def build_sparql_prompt_ablation(question, context):
    return (
        "Given the OWL model described in the following TTL file:\n"
        f"{context}\n"
        "Write a SPARQL query that answers the question. "
        "The data for your query is available in a SERVICE "
        "identified by <mapped>. Do not explain the query. "
        "return just the query, so it can be run verbatim "
        "from your response.\n"
        "Here’s the question:\n"
        f'"{question}"'
    )


# 5. Sinh prompt cho 43 câu hỏi × 2 phương pháp
prompt_rows = []

for _, row in baseline.iterrows():
    qid = int(row["ID"])
    method = row["Method"]
    question = row["Question"]
    context = contexts[qid][method]

    if method == "SQL":
        prompt = build_sql_prompt_ablation(question, context)
    else:
        prompt = build_sparql_prompt_ablation(question, context)

    prompt_rows.append({
        "ID": qid,
        "Method": method,
        "Question": question,
        "Prompt": prompt
    })

ablation_prompts = pd.DataFrame(prompt_rows)

assert len(ablation_prompts) == 86
assert not ablation_prompts["Prompt"].str.strip().eq("").any()

ablation_prompts.to_csv(
    OUT_DIR / "ablation_prompts.csv",
    index=False,
    encoding="utf-8-sig"
)

print("=== ABLATION PROMPTS ===")
print("Total prompts:", len(ablation_prompts))
print("SQL:", (ablation_prompts["Method"] == "SQL").sum())
print("SPARQL:", (ablation_prompts["Method"] == "SPARQL").sum())


# 7. Thống kê mức giảm ngữ cảnh
full_contexts = {
    "SQL": ddl_text,
    "SPARQL": OWL_FILE.read_text(encoding="utf-8")
}

context_stats = pd.DataFrame([
    {
        "ID": int(row["ID"]),
        "Method": row["Method"],
        "Context Ratio (%)": round(
            len(contexts[int(row["ID"])][row["Method"]])
            / len(full_contexts[row["Method"]]) * 100,
            2
        )
    }
    for _, row in ablation_prompts.iterrows()
])

print("\n=== CONTEXT REDUCTION ===")

display(
    context_stats.groupby("Method")["Context Ratio (%)"]
    .agg(["min", "mean", "max"])
    .round(2)
)

context_stats.to_csv(
    OUT_DIR / "context_reduction.csv",
    index=False
)

print("\nCell 3: COMPLETE")


print("\nSaved: ablation_prompts.csv")


Baseline notebook loaded: 10 cells
=== ABLATION PROMPTS ===
Total prompts: 86
SQL: 43
SPARQL: 43

=== CONTEXT REDUCTION ===


,min,mean,max
Method,,,
SPARQL,22.58,64.76,100.0
SQL,13.78,43.50,100.0



Cell 3: COMPLETE

Saved: ablation_prompts.csv


In [5]:
import copy
import contextlib
import io
import shutil

# 1. Cấu hình ablation
AB_RUN_IDS = range(1, 6)
AB_RUN_DIR = OUT_DIR / "generation_runs"
AB_RUN_DIR.mkdir(parents=True, exist_ok=True)


# 2. Nạp code từ notebook gốc với đúng thư mục làm việc
import os

original = {"__name__": "__main__", "display": display}

old_cwd = Path.cwd()

try:
    # Chuyển tạm về thư mục gốc của dự án
    os.chdir(PROJECT_DIR)

    for i, cell in enumerate(nb["cells"][:8], start=1):
        if cell["cell_type"] != "code":
            continue

        source = "".join(cell["source"])

        with contextlib.redirect_stdout(io.StringIO()):
            exec(
                compile(source, f"baseline_cell_{i}", "exec"),
                original
            )

finally:
    # Khôi phục thư mục làm việc ban đầu
    os.chdir(old_cwd)

print("Baseline components loaded successfully")
print("Working directory:", Path.cwd())


# 3. Kiểm tra các thành phần cần thiết
required = [
    "all_items",
    "prepare_kaggle_task",
    "kaggle_cli",
    "valid_csv",
    "required_columns",
    "BATCH_SIZE",
    "ddl_text",
    "owl_text"
]

missing = [name for name in required if name not in original]
assert not missing, f"Missing baseline components: {missing}"

# Đảm bảo nguồn ngữ cảnh khớp baseline
assert original["ddl_text"].strip() == ddl_text.strip(), (
    "DDL ablation khác DDL baseline"
)

assert original["owl_text"].strip() == full_contexts["SPARQL"].strip(), (
    "OWL ablation khác OWL baseline"
)

# 4. Thay prompt đầy đủ bằng prompt chọn lọc
ab_items = copy.deepcopy(original["all_items"])

prompt_map = {
    (int(row["ID"]), row["Method"]): row["Prompt"]
    for _, row in ablation_prompts.iterrows()
}

assert len(ab_items) == 86

for item in ab_items:
    key = (int(item["ID"]), item["Method"])
    prompt = prompt_map[key]

    # Hỗ trợ cả hai cách đặt tên trường
    item["Prompt"] = prompt
    item["prompt"] = prompt

# Trường hợp helper gốc gọi hàm build_prompt trực tiếp
question_prompts = {
    (row["Method"], row["Question"]): row["Prompt"]
    for _, row in ablation_prompts.iterrows()
}

original["build_sql_prompt"] = (
    lambda question: question_prompts[("SQL", question)]
)

original["build_sparql_prompt"] = (
    lambda question: question_prompts[("SPARQL", question)]
)

# Đổi thư mục output sang ablation
original["RUN_DIR"] = AB_RUN_DIR
original["all_items"] = ab_items

prepare_kaggle_task = original["prepare_kaggle_task"]
kaggle_cli = original["kaggle_cli"]
valid_csv = original["valid_csv"]

required_columns = original["required_columns"]
BATCH_SIZE = original["BATCH_SIZE"]

expected_keys = {
    (int(item["ID"]), item["Method"])
    for item in ab_items
}

print("=== ABLATION GENERATION ===")
print("Model:", MODEL)
print("Prompts:", len(ab_items))
print("Batch size:", BATCH_SIZE)
print("Pilot runs:", list(AB_RUN_IDS))
print("Output:", AB_RUN_DIR)

# 5. Chạy từng run
for run_id in AB_RUN_IDS:
    output_file = AB_RUN_DIR / f"generation_run_{run_id:02d}.csv"

    if output_file.exists():
        if valid_csv(output_file, expected_keys):
            print(f"Run {run_id:02d}: already completed")
            continue
        raise RuntimeError(f"Invalid run file: {output_file}")

    batch_dir = AB_RUN_DIR / f"kaggle_batches_run_{run_id:02d}"
    batch_dir.mkdir(parents=True, exist_ok=True)

    batch_files = []

    print(f"\n=== RUN {run_id:02d} ===", flush=True)

    for start in range(0, len(ab_items), BATCH_SIZE):
        batch_id = start // BATCH_SIZE + 1
        items = ab_items[start:start + BATCH_SIZE]

        task_name = f"acme-ctxsel-gpt54-r{run_id:02d}-b{batch_id:02d}"
        saved = batch_dir / f"{task_name}.csv"

        batch_keys = {
            (int(x["ID"]), x["Method"]) for x in items
        }

        if saved.exists():
            if valid_csv(saved, batch_keys):
                print(f"Batch {batch_id:02d}: reused")
                batch_files.append(saved)
                continue

            print(f"Batch {batch_id:02d}: invalid cache, regenerating")

            saved.unlink()

            download_dir = saved.parent / "downloads" / task_name
            if download_dir.exists():
                shutil.rmtree(download_dir)

        # Tạo task
        task_file = prepare_kaggle_task(task_name, items)

        # Gửi task và chạy GPT-5.4
        print(f"Batch {batch_id:02d}: push", flush=True)
        kaggle_cli("push", task_name, "-f", task_file, "--wait")

        print(f"Batch {batch_id:02d}: run", flush=True)
        kaggle_cli("run", task_name, "-m", MODEL, "--wait")

        # Tải kết quả
        download_dir = batch_dir / "downloads" / task_name
        kaggle_cli(
            "download", task_name,
            "-m", MODEL, "-o", download_dir
        )

        
        # Tìm và kiểm tra các kết quả Kaggle đã tải
        files = list(download_dir.rglob("generation.csv"))

        if not files:
            raise RuntimeError(f"Không tìm thấy generation.csv: {task_name}")

        valid_files = []

        for file in files:
            version = next(
                (int(p) for p in file.relative_to(download_dir).parts if p.isdigit()),
                0
            )

            is_valid = valid_csv(file, batch_keys)
            print(f"Version {version}: {'OK' if is_valid else 'INVALID'}")

            if is_valid:
                valid_files.append((version, file))

        if not valid_files:
            raise RuntimeError(f"Không có CSV hợp lệ: {task_name}")

        # Chọn phiên bản mới nhất trong các kết quả hợp lệ
        version, selected = max(valid_files, key=lambda x: (x[0], x[1].stat().st_mtime))

        result = pd.read_csv(selected, keep_default_na=False)
        result.to_csv(saved, index=False, encoding="utf-8-sig")

        assert valid_csv(saved, batch_keys)
        print(f"Batch {batch_id:02d}: sử dụng version {version}")


        batch_files.append(saved)
        print(f"Batch {batch_id:02d}: OK", flush=True)

    # 6. Gộp batch
    generation_df = pd.concat(
        [pd.read_csv(f, keep_default_na=False) for f in batch_files],
        ignore_index=True
    ).sort_values(["ID", "Method"]).reset_index(drop=True)

    generation_df[required_columns].to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    assert valid_csv(output_file, expected_keys)
    print(f"Run {run_id:02d}: COMPLETE (86/86)", flush=True)

print("\nAblation generation completed.")


,ID,Inquiry URI,Question
0,1,https://templates.data.world/IQ_03fc5a5571db11...,What is the loss ratio of each policy and agen...
1,2,https://templates.data.world/IQ_0d87a2a016062b...,Return all the policies we have by policy numb...
2,3,https://templates.data.world/IQ_1e19282f42eaea...,"What are the loss payment, loss reserve, expen..."
3,4,https://templates.data.world/IQ_1e57233cbc8a71...,"What are the total loss, which is the sum of l..."
4,5,https://templates.data.world/IQ_205ac9d7d19a51...,Return agents and the policy they have sold th...


Baseline components loaded successfully
Working directory: c:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\notebook\07_extended_analysis
=== ABLATION GENERATION ===
Model: gpt-5.4-2026-03-05
Prompts: 86
Batch size: 10
Pilot runs: [1, 2, 3, 4, 5]
Output: C:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\extended_analysis\context_ablation\generation_runs
Run 01: already completed
Run 02: already completed
Run 03: already completed
Run 04: already completed
Run 05: already completed

Ablation generation completed.


In [6]:
import pandas as pd
from pathlib import Path

# Đường dẫn kết quả Context Ablation
CA_DIR = PROJECT_DIR / "results/extended_analysis/context_ablation"
GEN_DIR = CA_DIR / "generation_runs"
EVAL_DIR = CA_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# Đọc toàn bộ 5 run
frames = []

for run_id in range(1, 6):
    files = list(GEN_DIR.glob(f"*run_{run_id:02d}.csv"))
    assert len(files) == 1, f"Không tìm thấy đúng một file Run {run_id:02d}: {files}"

    df = pd.read_csv(files[0], keep_default_na=False)
    df["ID"] = df["ID"].astype(int)
    df["Method"] = df["Method"].str.upper().str.strip()
    df["Run"] = run_id

    assert len(df) == 86
    assert not df.duplicated(["ID", "Method"]).any()
    assert df.groupby("Method").size().to_dict() == {"SQL": 43, "SPARQL": 43}
    assert df["Generated Query"].astype(str).str.strip().ne("").all()

    frames.append(df)

generation = pd.concat(frames, ignore_index=True)

# Kiểm tra cùng bộ câu hỏi giữa các run
keys = set(zip(frames[0]["ID"], frames[0]["Method"]))
for df in frames[1:]:
    assert set(zip(df["ID"], df["Method"])) == keys

assert len(generation) == 430

# Lưu dữ liệu đầu vào cho bước đánh giá
generation.to_csv(
    EVAL_DIR / "generation_all_runs.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Tổng lượt sinh:", len(generation))
print("Số run:", generation["Run"].nunique())
print("Thư mục đánh giá:", EVAL_DIR)

display(
    generation.groupby(["Run", "Method"])
    .agg(
        Questions=("ID", "count"),
        Generated=("Generated Query", lambda x: x.astype(str).str.strip().ne("").sum())
    )
)


Tổng lượt sinh: 430
Số run: 5
Thư mục đánh giá: C:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\extended_analysis\context_ablation\evaluation


Questions  Generated
Run Method                      
1   SPARQL         43         43
    SQL            43         43
2   SPARQL         43         43
    SQL            43         43
3   SPARQL         43         43
    SQL            43         43
4   SPARQL         43         43
    SQL            43         43
5   SPARQL         43         43
    SQL            43         43

In [7]:
import ast
import json
import re
import time
import duckdb
from collections import Counter
from decimal import Decimal

# Nạp hàm đánh giá từ notebook gốc
path = PROJECT_DIR / "notebook/05_evaluation.ipynb"
nb = json.loads(path.read_text(encoding="utf-8"))
funcs = {}

for cell in nb["cells"]:
    if cell["cell_type"] != "code":
        continue
    try:
        tree = ast.parse("".join(cell["source"]))
    except SyntaxError:
        continue
    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name in {"normalize", "same_result"}:
            funcs[node.name] = node

assert set(funcs) == {"normalize", "same_result"}

module = ast.fix_missing_locations(
    ast.Module(body=[funcs["normalize"], funcs["same_result"]], type_ignores=[])
)
env = {"pd": pd, "re": re, "Counter": Counter, "Decimal": Decimal}
exec(compile(module, "<evaluation>", "exec"), env)
same_result = env["same_result"]

# Đọc kết quả SQL tham chiếu
ref_dir = PROJECT_DIR / "results/sql/reference_results"
ref_map = pd.read_csv(
    PROJECT_DIR / "results/ontop/reference_comparison.csv",
    keep_default_na=False
)

references = {
    int(r["ID"]): ref_dir / (str(r["SQL Reference"]).split("/")[-1] + ".csv")
    for _, r in ref_map.iterrows()
}

# Chỉ đánh giá 215 truy vấn SQL
sql_data = generation[generation["Method"] == "SQL"].copy()
assert len(sql_data) == 215
assert not sql_data.duplicated(["Run", "ID"]).any()

out_dir = EVAL_DIR / "sql_results"
out_dir.mkdir(parents=True, exist_ok=True)
records = []

con = duckdb.connect(
    str(PROJECT_DIR / "results/sql/acme.duckdb"),
    read_only=True
)

try:
    for _, row in sql_data.iterrows():
        run_id, qid = int(row["Run"]), int(row["ID"])
        sql = str(row["Generated Query"]).strip()

        # Bỏ Markdown fence nếu có
        match = re.fullmatch(r"```(?:sql)?\s*\n(.*?)\n\s*```", sql, re.I | re.S)
        if match:
            sql = match.group(1).strip()

        folder = out_dir / f"run_{run_id:02d}"
        folder.mkdir(parents=True, exist_ok=True)
        output = folder / f"{qid:02d}.csv"

        status, correct, error = "ERROR", 0, ""
        start = time.perf_counter()
        output.unlink(missing_ok=True)

        try:
            # Chỉ chấp nhận truy vấn đọc
            assert re.match(r"^(SELECT|WITH)\b", sql, re.I)
            assert ";" not in sql.rstrip().rstrip(";")
            assert not re.search(
                r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE|ATTACH|COPY|EXPORT|INSTALL|LOAD)\b",
                sql, re.I
            )

            con.execute(sql).fetchdf().to_csv(output, index=False)
            correct = int(same_result(references[qid], output))
            status = "OK"

        except Exception as e:
            error = f"{type(e).__name__}: {e}"
            output.unlink(missing_ok=True)

        records.append({
            "Run": run_id,
            "ID": qid,
            "Method": "SQL",
            "Status": status,
            "Correct": correct,
            "Error": error,
            "Time (s)": round(time.perf_counter() - start, 3)
        })
finally:
    con.close()

sql_eval = pd.DataFrame(records)
sql_eval.to_csv(EVAL_DIR / "sql_evaluation.csv", index=False, encoding="utf-8-sig")

# Tính EA từng run
sql_summary = sql_eval.groupby("Run").agg(
    Questions=("ID", "count"),
    Executed=("Status", lambda x: (x == "OK").sum()),
    Correct=("Correct", "sum")
)

sql_summary["EA (%)"] = (sql_summary["Correct"] / 43 * 100).round(2)

display(sql_summary)
print("Đã lưu:", EVAL_DIR / "sql_evaluation.csv")


,Questions,Executed,Correct,EA (%)
Run,,,,
1,43,19,11,25.58
2,43,19,10,23.26
3,43,19,11,25.58
4,43,20,11,25.58
5,43,18,8,18.60


Đã lưu: C:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\extended_analysis\context_ablation\evaluation\sql_evaluation.csv


In [16]:

import re
import time
import subprocess
import pandas as pd
from pathlib import Path
from datetime import datetime

# 1. Cấu hình Ontop và thư mục kết quả
ONTOP = Path.home() / "tools/ontop/ontop.bat"
MAPPING = PROJECT_DIR / "results/ontop/acme_mapping.ttl"
ONTOLOGY = PROJECT_DIR / "data/ACME_Insurance/ontology/insurance.ttl"
PROPERTIES = PROJECT_DIR / "results/ontop/duckdb.properties"

SPARQL_REF_DIR = PROJECT_DIR / "results/ontop/reference_results"
SPARQL_OUT_DIR = EVAL_DIR / "sparql_results"
SPARQL_OUT_DIR.mkdir(parents=True, exist_ok=True)

detail_path = EVAL_DIR / "sparql_evaluation.csv"

for path in [ONTOP, MAPPING, ONTOLOGY, PROPERTIES]:
    assert path.exists(), f"Thiếu file: {path}"

assert callable(same_result), "Chưa nạp hàm same_result()"

# 2. Chuẩn bị 215 truy vấn SPARQL
sparql_data = generation[generation["Method"] == "SPARQL"].copy()
sparql_data = sparql_data.sort_values(["Run", "ID"])

assert len(sparql_data) == 215
assert not sparql_data.duplicated(["Run", "ID"]).any()

for qid in sparql_data["ID"].unique():
    name = "18_ontop.csv" if qid == 18 else f"{qid:02d}.csv"
    assert (SPARQL_REF_DIR / name).exists(), f"Thiếu Reference: {name}"

# 3. Đọc checkpoint và loại bỏ lỗi đường dẫn cũ
if detail_path.exists():
    saved = pd.read_csv(detail_path, keep_default_na=False)

    old_errors = saved["Error"].str.contains(
        r"Invalid (?:property|mapping) file path|Win%20",
        case=False,
        regex=True,
        na=False
    )

    if old_errors.any():
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup = EVAL_DIR / f"sparql_evaluation_backup_{timestamp}.csv"
        saved.to_csv(backup, index=False, encoding="utf-8-sig")

        print(f"Đã sao lưu checkpoint: {backup.name}")
        print(f"Loại bỏ {old_errors.sum()} kết quả lỗi đường dẫn cũ")

        saved = saved.loc[~old_errors].copy()

    records = saved.to_dict("records")
else:
    records = []

completed = {
    (int(r["Run"]), int(r["ID"])): r
    for r in records
}

# 4. Thực thi và đánh giá từng truy vấn
for _, row in sparql_data.iterrows():
    run_id = int(row["Run"])
    qid = int(row["ID"])
    query = str(row["Generated Query"]).strip()
    key = (run_id, qid)

    # Chỉ bỏ qua nếu truy vấn đã được đánh giá với cùng nội dung
    if key in completed:
        if str(completed[key]["Generated Query"]) == query:
            continue

        records = [
            r for r in records
            if (int(r["Run"]), int(r["ID"])) != key
        ]

    folder = SPARQL_OUT_DIR / f"run_{run_id:02d}"
    folder.mkdir(parents=True, exist_ok=True)

    query_file = folder / f"{qid:02d}_ontop.rq"
    output_file = folder / f"{qid:02d}.csv"

    ref_name = "18_ontop.csv" if qid == 18 else f"{qid:02d}.csv"
    ref_file = SPARQL_REF_DIR / ref_name

    status = "ERROR"
    correct = 0
    error = ""
    start = time.perf_counter()

    output_file.unlink(missing_ok=True)

    try:
        # Xóa Markdown fence
        clean_query = re.sub(
            r"(?im)^\s*```(?:sparql)?\s*$|^\s*```\s*$",
            "",
            query
        ).strip()

        # Ontop local không sử dụng SERVICE <mapped>
        clean_query = re.sub(
            r"\bSERVICE\s*<mapped>\s*(?=\{)",
            "",
            clean_query,
            flags=re.IGNORECASE
        )

        if not re.search(r"\bSELECT\b", clean_query, re.I):
            raise ValueError("Không tìm thấy SELECT")

        query_file.write_text(clean_query, encoding="utf-8")

        # Thực thi với đường dẫn đã kiểm tra thành công
        result = subprocess.run(
            [
                str(ONTOP), "query",
                "-m", MAPPING.resolve().as_uri(),
                "-t", str(ONTOLOGY.resolve()),
                "-p", PROPERTIES.resolve().as_uri(),
                "-q", str(query_file.resolve()),
                "-o", str(output_file)
            ],
            cwd=ONTOP.parent,
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
            timeout=120
        )

        log = result.stderr + result.stdout

        # Kiểm tra cả mã thoát, log và file kết quả
        if result.returncode != 0 or not output_file.exists():
            raise RuntimeError(
                log[-2000:] or "Ontop không tạo CSV"
            )

        if "Exception in thread" in log or "RepositoryException" in log:
            raise RuntimeError(log[-2000:])

        status = "OK"

        # So sánh kết quả với Reference
        try:
            correct = int(same_result(ref_file, output_file))
        except Exception as e:
            status = "EVAL_ERROR"
            error = f"{type(e).__name__}: {e}"

    except Exception as e:
        error = f"{type(e).__name__}: {e}"

    records.append({
        "Run": run_id,
        "ID": qid,
        "Method": "SPARQL",
        "Generated Query": query,
        "Status": status,
        "Correct": correct,
        "Error": error,
        "Time (s)": round(time.perf_counter() - start, 3)
    })

    # Lưu checkpoint sau mỗi truy vấn
    pd.DataFrame(records).to_csv(
        detail_path,
        index=False,
        encoding="utf-8-sig"
    )

    completed[key] = records[-1]

    print(
        f"Run {run_id:02d} | Q{qid:02d} | "
        f"{status} | Correct={correct}"
    )

# 5. Tổng hợp Execution Accuracy
sparql_eval = pd.DataFrame(records)

assert len(sparql_eval) == 215
assert not sparql_eval.duplicated(["Run", "ID"]).any()
assert not sparql_eval["Status"].eq("EVAL_ERROR").any(), \
    "Có lỗi khi so sánh Reference, cần kiểm tra log"

sparql_summary = sparql_eval.groupby("Run").agg(
    Questions=("ID", "count"),
    Executed=("Status", lambda x: (x == "OK").sum()),
    Correct=("Correct", "sum")
)

sparql_summary["EA (%)"] = (
    sparql_summary["Correct"] / sparql_summary["Questions"] * 100
).round(2)

display(sparql_summary)

print("\nTổng truy vấn:", len(sparql_eval))
print("Thực thi thành công:", (sparql_eval["Status"] == "OK").sum())
print("Trả lời đúng:", sparql_eval["Correct"].sum())
print("Đã lưu:", detail_path)

Đã sao lưu checkpoint: sparql_evaluation_backup_20261114_233933.csv
Loại bỏ 215 kết quả lỗi đường dẫn cũ
Run 01 | Q01 | OK | Correct=0
Run 01 | Q02 | OK | Correct=1
Run 01 | Q03 | OK | Correct=1
Run 01 | Q04 | OK | Correct=1
Run 01 | Q05 | OK | Correct=1
Run 01 | Q06 | ERROR | Correct=0
Run 01 | Q07 | ERROR | Correct=0
Run 01 | Q08 | OK | Correct=0
Run 01 | Q09 | OK | Correct=1
Run 01 | Q10 | OK | Correct=1
Run 01 | Q11 | OK | Correct=1
Run 01 | Q12 | OK | Correct=1
Run 01 | Q13 | ERROR | Correct=0
Run 01 | Q14 | OK | Correct=0
Run 01 | Q15 | OK | Correct=1
Run 01 | Q16 | OK | Correct=1
Run 01 | Q17 | OK | Correct=1
Run 01 | Q18 | ERROR | Correct=0
Run 01 | Q19 | OK | Correct=1
Run 01 | Q20 | OK | Correct=1
Run 01 | Q21 | OK | Correct=1
Run 01 | Q22 | OK | Correct=1
Run 01 | Q23 | OK | Correct=1
Run 01 | Q24 | OK | Correct=0
Run 01 | Q25 | OK | Correct=1
Run 01 | Q26 | OK | Correct=1
Run 01 | Q27 | OK | Correct=0
Run 01 | Q28 | OK | Correct=1
Run 01 | Q29 | OK | Correct=1
Run 01 | Q30 

,Questions,Executed,Correct,EA (%)
Run,,,,
1,43,36,28,65.12
2,43,36,25,58.14
3,43,38,30,69.77
4,43,37,29,67.44
5,43,35,25,58.14



Tổng truy vấn: 215
Thực thi thành công: 182
Trả lời đúng: 137
Đã lưu: C:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\extended_analysis\context_ablation\evaluation\sparql_evaluation.csv


In [17]:
import pandas as pd

# 1. Đọc kết quả đánh giá SQL và SPARQL
sql_eval = pd.read_csv(
    EVAL_DIR / "sql_evaluation.csv",
    keep_default_na=False
)

sparql_eval = pd.read_csv(
    EVAL_DIR / "sparql_evaluation.csv",
    keep_default_na=False
)

# 2. Gộp và kiểm tra dữ liệu
evaluation = pd.concat(
    [sql_eval, sparql_eval],
    ignore_index=True
)

evaluation["Correct"] = pd.to_numeric(
    evaluation["Correct"], errors="raise"
).astype(int)

evaluation["Run"] = evaluation["Run"].astype(int)
evaluation["ID"] = evaluation["ID"].astype(int)

assert len(evaluation) == 430
assert not evaluation.duplicated(["Method", "Run", "ID"]).any()
assert evaluation["Correct"].isin([0, 1]).all()
assert not evaluation["Status"].eq("EVAL_ERROR").any()

assert set(evaluation["Method"]) == {"SQL", "SPARQL"}

for method, df in evaluation.groupby("Method"):
    assert len(df) == 215
    assert set(df["Run"]) == set(range(1, 6))
    assert df.groupby("Run")["ID"].nunique().eq(43).all()
    assert df.groupby("ID")["Run"].nunique().eq(5).all()

# 3. EA: tỷ lệ truy vấn đúng trong mỗi run
ea = evaluation.groupby(["Method", "Run"]).agg(
    Questions=("ID", "count"),
    Executed=("Status", lambda x: (x == "OK").sum()),
    Correct=("Correct", "sum")
).reset_index()

ea["EA (%)"] = ea["Correct"] / ea["Questions"] * 100

# 4. OEA: tỷ lệ đúng của từng câu hỏi qua 5 run
oea = evaluation.groupby(["Method", "ID"]).agg(
    Runs=("Run", "count"),
    Correct=("Correct", "sum")
).reset_index()

oea["OEA (%)"] = oea["Correct"] / oea["Runs"] * 100

# 5. AOEA: trung bình OEA của 43 câu hỏi
aoea = oea.groupby("Method").agg(
    Questions=("ID", "count"),
    AOEA=("OEA (%)", "mean")
).reset_index()

aoea = aoea.rename(columns={"AOEA": "AOEA (%)"})

# 6. Lưu kết quả để sử dụng trong báo cáo
evaluation.to_csv(
    EVAL_DIR / "evaluation_all.csv",
    index=False, encoding="utf-8-sig"
)

ea.to_csv(
    EVAL_DIR / "ea_by_run.csv",
    index=False, encoding="utf-8-sig"
)

oea.to_csv(
    EVAL_DIR / "oea_by_question.csv",
    index=False, encoding="utf-8-sig"
)

aoea.to_csv(
    EVAL_DIR / "aoea_summary.csv",
    index=False, encoding="utf-8-sig"
)

# 7. Hiển thị kết quả
print("EA theo từng run:")
display(ea.round(2))

print("\nAOEA tổng hợp:")
display(aoea.round(2))

print("\nĐã đánh giá:", len(evaluation), "/ 430 truy vấn")
print("Đã lưu kết quả tại:", EVAL_DIR)


EA theo từng run:


,Method,Run,Questions,Executed,Correct,EA (%)
0,SPARQL,1,43,36,28,65.12
1,SPARQL,2,43,36,25,58.14
2,SPARQL,3,43,38,30,69.77
3,SPARQL,4,43,37,29,67.44
4,SPARQL,5,43,35,25,58.14
5,SQL,1,43,19,11,25.58
6,SQL,2,43,19,10,23.26
7,SQL,3,43,19,11,25.58
8,SQL,4,43,20,11,25.58
9,SQL,5,43,18,8,18.60



AOEA tổng hợp:


,Method,Questions,AOEA (%)
0,SPARQL,43,63.72
1,SQL,43,23.72



Đã đánh giá: 430 / 430 truy vấn
Đã lưu kết quả tại: C:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\extended_analysis\context_ablation\evaluation
